# Multivariate Analysis: Decoding MEG/EEG data

`Authors: Natalie Schaworonkow, Simon Kern, Marijn van Vliet, Britta Westner, Alexandre Gramfort`


Let's see if we can do some _single trial_ decoding of our data! 

For that, we will use the MNE-Python decoding module, which makes use of the extensive machine learning library and toolbox [scikit-learn](https://scikit-learn.org/stable/).

`
Reference:
Scikit-learn: Machine Learning in Python,
Pedregosa et al., JMLR 12, pp. 2825-2830, 2011.
`

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import mne
import numpy as np

mne.set_log_level('error')

# Change the following path to where the folder ds000117 is on your disk.
data_path = "../ds000117_pruned"  # `./` means the folder of this notebook

# Change the following path to where you unzipped the extra data (`extra_meg_data-v2.zip`) on your disk.
extra_path = "../extra_data_mne"  # `./` means the folder of this notebook

## Load the epochs we saved

In [ ]:
epochs_fname = f"{data_path}/derivatives/meg_derivatives/sub-01/ses-meg/meg/sub-01_ses-meg_task-facerecognition_run-01_proc-sss-epo.fif"
epochs = mne.read_epochs(epochs_fname)
epochs.crop(0, 0.4)
epochs.pick_types(meg='mag')

Let's compute the ERFs and look at the contrast between _face_ and _scrambled_ responses.

In [ ]:
evoked_face = epochs["face"].average()
evoked_scrambled = epochs["scrambled"].average()
evoked_contrast = mne.combine_evoked([evoked_face, evoked_scrambled], [0.5, -0.5])

Let's check the signal ...

In [ ]:
# Fit a sphere to the headshape in order to make proper topo plots.
# This is needed for this particular dataset and may not be necessary for yours.
# I left this in mainly because maybe you want to check the decoding for EEG.
radius, center, _ = mne.bem.fit_sphere_to_headshape(epochs.info, dig_kinds="eeg")
sphere = tuple(center) + (radius,)

evoked_face.plot(sphere=sphere);
evoked_scrambled.plot(sphere=sphere);
evoked_contrast.plot(sphere=sphere);

... and plot some topographies:

In [ ]:
vmax = 150  #4
evoked_face.plot_topomap(contours=0, vlim=(0, vmax), sphere=sphere);
evoked_scrambled.plot_topomap(contours=0, vlim=(0, vmax), sphere=sphere);

vmax = 80  #2
evoked_contrast.plot_topomap(contours=0, vlim=(0, vmax), sphere=sphere);

## Let's prepare our data for classification

First, we equalize the trials per condition, so 50% is actually our chance level:

In [ ]:
epochs.equalize_event_counts(["face", "scrambled"])

For classification, we first need a _response_ vector. In our case, that is a vector that describes for each trial, whether the trial was 
a _face_ or a _scrambled_ trial. 

In [ ]:
epochs.events

In [ ]:
epochs.event_id

We set _face_ to 1, and _scrambled_ to 0.

In [ ]:
# initialize with zeros and then set faces to 1
y = np.zeros(len(epochs.events), dtype=int)
y[epochs.events[:, 2] < 17] = 1  # 1 means face

y.size # check the length

We also need an array that contains our data (_observations x channels x time_):

In [ ]:
X = epochs.get_data()
X.shape

## Let's see if we can classify the data based on single trials

In [ ]:
# we use a logistic regression model and import this from scikit learn:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression

# We then set the parameters (regularization value and solver):
lda = LinearDiscriminantAnalysis()
logreg = LogisticRegression(C=1e6, solver='liblinear')  # alternative classifier

# We wrap the scikit-learn model in a mne.decoder.LinearModel class
# for extracting patterns later.
from mne.decoding import LinearModel
lm = LinearModel(lda)  # For the exercise: try the logreg model

Let's use the `mne.decoding` module to manage our classification:

In [ ]:
from sklearn.pipeline import make_pipeline
from mne.decoding import Scaler, Vectorizer, cross_val_multiscore

# We make a pipeline object that specifies how to handle the data:
clf = make_pipeline(Scaler(epochs.info),  # Scale 
                    Vectorizer(), # Go from shape=(epochs, channels, time) to shape=(epochs, channels * time)
                    lm) # Machine learning algorithm

# Now let's run the classification and score based on a 5-fold-cross-validation:
scores = cross_val_multiscore(clf, X, y, cv=5, n_jobs=1)

# Let's compute the mean scores across cross-validation splits:
score = np.mean(scores, axis=0)
print(f"Spatio-temporal: {100 * score}%")

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Try the decoding with Logistic Regression instead of LDA.</li>
    </ul>
</div>

## Visualizing filters and patterns

The `LinearModel` class of MNE-Python performs the Haufe transform on the linear model, giving us access to both the filters (=the regression coefficients) and the patterns. Let's pack them into MNE-Python objects and visualize them.

In [ ]:
# Train the model of all data (no cross-validation)
clf.fit(X, y)

In [ ]:
# plot same timepoints for both patterns & filters
times= (0.2, 0.275, 0.3, 0.35)
# Visualize the filters
filters = lm.filters_.reshape(len(epochs.ch_names), len(epochs.times))
filters = mne.EvokedArray(filters, epochs.info, tmin=epochs.times[0])
filters.plot_joint(times=times);



In [ ]:
# Visualize the patterns
patterns_data = lm.patterns_.reshape(len(epochs.ch_names), len(epochs.times))
patterns = mne.EvokedArray(patterns_data, epochs.info, tmin=epochs.times[0])
patterns.plot_joint(times=times);

In [ ]:
# next, create an animation
import matplotlib.animation
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import zscore
#get the sensor positions
plt.rcParams["animation.html"] = "jshtml"
plt.ioff()
fig, ax = plt.subplots(figsize=[3, 3])
# zscore as the values are otherwise to small to display

def animate(t):
    ax.clear()
    mne.viz.plot_topomap(patterns_data[:, t], pos=epochs.info, axes=ax, show=False,
                         vlim=[np.min(patterns_data), np.max(patterns_data)])
    ax.set_title(f'spatial pattern at {epochs.times[t]:.2f}s')
matplotlib.animation.FuncAnimation(fig, animate, frames=patterns_data.shape[1])


<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Compare the weights and the patterns with the contrast evoked plot we made earlier. Anything of note?</li>
    </ul>
</div>

## Simple spatial patterns vs. spatial filters demo

In [ ]:
# define a spatial filter that just corresponds to a single channel
picks = mne.pick_channels(epochs.ch_names,
                          ['MEG0421', 'MEG0411', 'MEG0641', 'MEG0431', 'MEG0631'],
                         ordered=True)
spatial_filter = np.zeros((len(epochs.ch_names)))
spatial_filter[picks[0]] = 1

# remove the comment to create a spatial filter that subtracts the 
# average of 4 surrounding electrodes from the central electrodes
# = Laplacian filter

# spatial_filter[picks[1:]] = -.25 

# compute the corresponding spatial pattern
cov_data = mne.compute_covariance(epochs).data
spatial_pattern = spatial_filter.T @ cov_data

fig, ax = plt.subplots(1, 2, figsize=(5,5))
mne.viz.plot_topomap(spatial_filter, epochs.info, axes=ax[0], show=False);
mne.viz.plot_topomap(spatial_pattern, epochs.info, axes=ax[1], show=False);
# mne.viz.plot_topomap(cov_data[20], epochs.info, axes=ax[1], show=False);
ax[0].set(title='spatial filter')
ax[1].set(title='spatial pattern')

plt.show()

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Instead of taking the channel as is, look at the impact of subtracting neighboring electrodes on spatial pattern, by commenting in the corresponding line and comparing the spatial patterns </li>
      <li> You can also select another central electrode and subtract the neighboring electrodes. (How would you find the neighbor channel names?)</li> 
    </ul>
</div>

## Statistical testing

With accuracies >70% we can be pretty sure the model is performing better than chance. But let's formally test this against models with shuffled class labels that truly are performing at chance level.

In [ ]:
from tqdm import trange

rng = np.random.default_rng(seed=1)

n = 100  # Numer of random models. Increase for better p-value estimation.
cv = 2  # Number of cross-validation folds. We decrease it here to save some time.
X_restricted = epochs.copy().crop(0.15, 0.4).get_data()  # restrict the data to save time.

random_scores = list()
y_random = y.copy()
for _ in trange(n):
    rng.shuffle(y_random)  # shuffles in-place
    score = cross_val_multiscore(clf, X_restricted, y_random, cv=cv, n_jobs=1).mean()
    random_scores.append(score)
random_scores = np.array(random_scores)

actual_score = cross_val_multiscore(clf, X_restricted, y, cv=cv, n_jobs=1).mean()

In [ ]:
# Plot the distribution of random scores and annotate with the actual score.
plt.figure(figsize=(8, 4))
plt.hist(random_scores, label="random scores")
plt.axvline(actual_score, color="red", label="actual score")
plt.legend()

To compute a p-value, count the number of times a random model's performance exceeded our actual performance:

In [ ]:
pvalue = (random_scores > actual_score).mean()
print("p-value:", pvalue)

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Try plotting the patterns, and performing the statistics with Logistic Regression instead of LDA.</li>
      <li>What do you think about p-value = 0 (zero)? Good/bad?</li>
    </ul>
</div>

## Decoding over time

Often more interesting for electrophysiological data is to keep the time dimension and to slide
the decoding model across time:

In [ ]:
from sklearn.preprocessing import StandardScaler
from mne.decoding import SlidingEstimator

# We prepare our pipeline again:
clf = make_pipeline(StandardScaler(), lda)

# But now wrap this pipeline in a sliding estimator:
time_decod = SlidingEstimator(clf, n_jobs=1, scoring="roc_auc", verbose=True)

# Let's score it:
scores = cross_val_multiscore(time_decod, X, y, cv=10, n_jobs=1)

# Mean scores across cross-validation splits:
scores_mean = np.mean(scores, axis=0)
scores_std = np.std(scores, axis=0)

# Let's see the dimesion of our scores:
scores_mean.shape


We kept the time dimension - by fitting one model per time-point. Let's plot our scores!

In [ ]:
# prepare plot

fig, ax = plt.subplots()

# plot scores and chance level
ax.plot(epochs.times, scores_mean, label="score")
ax.fill_between(epochs.times, scores_mean - scores_std, scores_mean + scores_std, alpha=0.5)
ax.axhline(.5, color='k', linestyle="--", label="chance")
ax.axvline(.0, color='k', linestyle="-")  # mark time 0

# set axis labels, legend, title
ax.set_xlabel("Times")
ax.set_ylabel("AUC")  # Area Under the Curve
ax.legend()

ax.set_title("Sensor space decoding")

plt.show()

<div class='alert alert-success'>
    <b>EXERCISE</b>:
     <ul>
      <li>Try the decoding over time with Logistic Regression instead of LDA.</li>
    </ul>
</div>

## Temporal Generalization

Next let's see how our stimulus decoding generalizes across time! That means, we train a decoder on a specific timepoint and test it on another. This can be useful to see how temporally stable a pattern (a common brain activity pattern that is more stable across time) is or if there is an unfolding of different patterns over time.


In [ ]:
from mne.decoding import GeneralizingEstimator
 
clf = make_pipeline(
    StandardScaler(),
    lda,
)
 
time_gen = GeneralizingEstimator(clf, scoring="roc_auc", n_jobs=None, verbose=True)
 
time_gen.fit(X=X, y=y)
scores = time_gen.score(X=X, y=y)
 


Next let plot the results! And see how temporally generalizable the pattern is that is learned.

In [ ]:
fig, ax = plt.subplots(layout="constrained")
im = ax.matshow(
    scores,
    vmin=0,
    vmax=1.0,
    cmap="RdBu_r",
    origin="lower",
    extent=epochs.times[[0, -1, 0, -1]],
)
ax.xaxis.set_ticks_position("bottom")
ax.set_xlabel('Testing time (s)')
ax.set_ylabel('Training time (s)')
ax.set_title("Generalization across time")
fig.colorbar(im, ax=ax, label="Performance (ROC AUC)")
plt.show()

It seems like after ~200ms, the brain activity pattern also generalizes to other time points! However, we also see a diagonal. That is, because we are training and testing on the same data. For this reason, the classifier is perfectly able to predict the class, as it has been trained on exactly that data. 😬 To prevent that, we would need to use a cross-validation approach we did above: Train on a subset of the data and then predict the remaining data. But we skip it here because of computation time.

## Spatial filtering: SSD

Spatial filtering provides a way to efficiently perform dimensionality reduction. Here, we use a technique called Spatio-Spectral Decomposition (SSD) to extract time courses with maximal alpha-activity. For example here, we want to maximize activity from 8–12 Hz and minimize activity in the flanking bands, around 7 and around 13 Hz. But we could use any frequency band, depends on our interest.

<div>
<img src="https://i.imgur.com/5oYQiR9.png" width="300"/>
</div>


In [ ]:
# read-in epochs again because we want a bit more data than our cropped interval from above
epochs_fname = f"{data_path}/derivatives/meg_derivatives/sub-01/ses-meg/meg/sub-01_ses-meg_task-facerecognition_run-01_proc-sss-epo.fif"
epochs = mne.read_epochs(epochs_fname)
epochs.pick_types(meg='mag')


from mne.decoding import SSD

# define our frequency band of interest
alpha_min, alpha_max = 8, 12
freqs_noise = 7, 13
ssd = SSD(
    info=epochs.info,
    reg="oas",
    filt_params_signal=dict(
        l_freq=alpha_min,
        h_freq=alpha_max,
        l_trans_bandwidth=1,
        h_trans_bandwidth=1,
    ),
    filt_params_noise=dict(
        l_freq=freqs_noise[0],
        h_freq=freqs_noise[1],
        l_trans_bandwidth=1,
        h_trans_bandwidth=1,
    ),
)
ssd.fit(X=epochs.get_data())

After we performed the decomposition, we obtain the traces using the SSD spatial filters, also called components sometimes (similar as ICA). The components are ordered by alpha-SNR, such that the first one has maximum alpha-SNR, and the last one has minimum alpha-SNR. We inspect the spectra of the signals.

In [ ]:
%matplotlib inline
ssd_sources = ssd.transform(X=epochs.get_data())
print(ssd_sources.shape) # we get as many components as sensors (102)


# Get psd of SSD-filtered signals.
psd, freqs = mne.time_frequency.psd_array_welch(
    ssd_sources, sfreq=epochs.info["sfreq"], n_fft=512,
    fmax=45
)

fig, ax = plt.subplots(1)
ax.loglog(freqs, psd[:, 0].mean(axis=0), label="component with maximal alpha-SNR")
ax.loglog(freqs, psd[:, -1].mean(axis=0), label="component with minimum alpha-SNR")

# for highlighting the freq. band of interest
bandfilt = (8 <= freqs) & (freqs <= 12)
ax.fill_between(freqs[bandfilt], 0, 10000, color="green", alpha=0.15)
ax.set_xlabel("log (frequency)")
ax.set_ylabel("log (power)")
ax.legend()
fig.show()

#

We plot the example traces and the corresponding topography of the first component. We obtained a nice parietal alpha source! We could now go on and compute time-frequency plots, just using this one component. But it's the end of the week! You made it. :)

In [ ]:
i_comp = 0
fig, ax = plt.subplots(1, 2, figsize=(8, 10), width_ratios=[2, 1])
n_trials = 20
for i in range(n_trials):
    ax[0].plot(epochs.times, ssd_sources[i][i_comp] + 40*i, 'k')
ax[0].axvline(0, c='r')
mne.viz.plot_topomap(ssd.patterns_[i_comp], epochs.info, axes=ax[1], show=False)
fig.show()

## Further reading

For more details see: https://mne.tools/stable/auto_tutorials/machine-learning/50_decoding.html

and this book chapter:
```
Jean-Rémi King, Laura Gwilliams, Chris Holdgraf, Jona Sassenhagen, Alexandre Barachant, Denis Engemann, Eric Larson, Alexandre Gramfort. Encoding and Decoding Neuronal Dynamics: Methodological Framework to Uncover the Algorithms of Cognition. 2018. https://hal.archives-ouvertes.fr/hal-01848442/
```